# Retrieval Agent With Custom Tools

In this notebook, I build a retrieval agent using the Hugging Face `smolagents` library.

The agent answers questions about the guests attending a party by searching through a small knowledge base that I store directly in the notebook. This is a simple form of retrieval: instead of the model guessing an answer, the agent calls a tool that looks up the real information first.

In this notebook, I will learn how to:

- Store a small knowledge base as plain Python data
- Write a keyword based retriever without using any embedding model
- Turn the retriever into an agent tool using the `@tool` decorator
- Rebuild the same tool as a class that inherits from `Tool`
- Combine the retriever with a web search tool so the agent can fall back to the internet
- Let the agent decide which tool to use for each question

I keep everything lightweight on purpose. There is no vector database and no model running on my machine, so the notebook stays fast to run.

## 1. Importing Libraries and Creating the Model

First I import the pieces I need from `smolagents`.

`CodeAgent` is the agent that writes and runs Python code to solve a task. `tool` and `Tool` are the two ways of creating a custom tool. `InferenceClientModel` is the language model, which runs on the Hugging Face Inference API rather than on my own machine.

In [ ]:
from smolagents import (
    CodeAgent,
    DuckDuckGoSearchTool,
    InferenceClientModel,
    Tool,
    tool
)

model = InferenceClientModel(
    model_id="Qwen/Qwen2.5-Coder-32B-Instruct"
)

## 2. Building the Guest Knowledge Base

The knowledge base is the information the agent is allowed to look things up in. In a real project this would be a database or a set of documents, but here a list of dictionaries is enough.

Each guest has a name, a relation, a short description and the year they were born. The language model has no way of knowing any of this, so the only way the agent can answer correctly is by calling my retriever tool.

In [ ]:
guests = [
    {
        "name": "Ada Lovelace",
        "relation": "best friend",
        "description": "A respected mathematician and writer, known for her work on Charles Babbage's Analytical Engine. She is often described as the first computer programmer.",
        "born": 1815,
    },
    {
        "name": "Alan Turing",
        "relation": "old university contact",
        "description": "A mathematician and logician who formalised the idea of computation. He enjoys long walks and does not enjoy small talk.",
        "born": 1912,
    },
    {
        "name": "Grace Hopper",
        "relation": "colleague from the navy",
        "description": "A computer scientist and rear admiral who worked on the first compilers. She prefers her coffee black and her meetings short.",
        "born": 1906,
    },
    {
        "name": "Marie Curie",
        "relation": "distant relative",
        "description": "A physicist and chemist who carried out pioneering research on radioactivity. She is the only person to win a Nobel Prize in two different sciences.",
        "born": 1867,
    },
]

print(f"The knowledge base contains {len(guests)} guests.")

## 3. Turning a Guest Record Into Readable Text

A tool has to return text, because the text is what the language model actually reads.

So before writing the retriever I write a small helper that turns one guest dictionary into a clean, readable paragraph. Keeping this in its own function means the retriever stays short and I can change the wording in one place.

In [ ]:
def format_guest(guest: dict) -> str:
    """Turns a single guest record into a readable block of text."""
    return (
        f"Name: {guest['name']}\n"
        f"Relation: {guest['relation']}\n"
        f"Born: {guest['born']}\n"
        f"Description: {guest['description']}"
    )


print(format_guest(guests[0]))